In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

## Generate Similarity Matrix

USM contains the following 
### One Hot Categoricals
1. Airing Status
2. Genres
3. Themes
4. Demographics
5. Rating
6. Source

### Continuous Numerics
1. Score
2. Members
3. Favorites
4. Related_Entries
5. Year
6. Episodes

Missing values are present in Score and Episodes.

In [2]:
USM = pd.read_csv('USM.csv', index_col=0)
USM = USM.loc[~USM.index.duplicated(keep='first')] # Duplicates present from shows continued airing between multiple seasons but still classified as one.
USM.describe()

,Is_Finished,Genre_Action,Genre_Adventure,Genre_Supernatural,Genre_Mystery,Genre_Sci-Fi,Genre_Suspense,Genre_Comedy,Genre_Romance,Genre_Drama,...,Source_4-koma manga,Source_Light novel,Source_Manga,Source_Web manga,Score,Members,Favorites,Related_Entries,Year,Episodes
count,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,10118.000000,...,10118.000000,10118.000000,10118.000000,10118.000000,7447.000000,1.011800e+04,10118.000000,10118.000000,10118.000000,9928.000000
mean,0.974007,0.260723,0.162680,0.070666,0.047045,0.114252,0.025697,0.353726,0.086875,0.099328,...,0.021546,0.070271,0.234236,0.047836,6.660481,7.438278e+04,782.378039,1.253805,2017.935857,12.732474
std,0.159123,0.439051,0.369092,0.256279,0.211745,0.318133,0.158237,0.478149,0.281666,0.299117,...,0.145202,0.255615,0.423541,0.213429,0.866052,2.336106e+05,5697.289173,1.254909,3.622431,36.318775
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,2.400000,3.500000e+01,0.000000,0.000000,2012.000000,1.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,6.065000,4.540000e+02,0.000000,0.000000,2015.000000,1.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,6.650000,4.382000e+03,7.000000,2.000000,2018.000000,6.000000
75%,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,7.270000,3.713275e+04,87.000000,2.000000,2021.000000,13.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,9.310000,4.119103e+06,234614.000000,5.000000,2024.000000,1818.000000


Add scaling to avoid numerics from overpowering booleans.
We will be imputing with 0.

In [3]:
scaler = MinMaxScaler()

cols = ['Score', 'Members', 'Favorites', 'Related_Entries', 'Year', 'Episodes']
USM[cols] = scaler.fit_transform(USM[cols])

USM = USM.fillna(0)
USM

,Title,Is_Finished,Genre_Action,Genre_Adventure,Genre_Supernatural,Genre_Mystery,Genre_Sci-Fi,Genre_Suspense,Genre_Comedy,Genre_Romance,...,Source_4-koma manga,Source_Light novel,Source_Manga,Source_Web manga,Score,Members,Favorites,Related_Entries,Year,Episodes
MAL_id,,,,,,,,,,,,,,,,,,,,,
14719,JoJo no Kimyou na Bouken (TV),1,1,1,1,0,0,0,0,0,...,0,0,1,0,0.791606,0.423153,0.162902,0.8,0.0,0.013759
13601,Psycho-Pass,1,1,0,0,1,1,1,0,0,...,0,0,0,0,0.858177,0.409326,0.168421,0.4,0.0,0.011558
14741,Chuunibyou demo Koi ga Shitai!,1,0,0,0,0,0,0,1,1,...,0,1,0,0,0.767004,0.340986,0.078197,0.4,0.0,0.006054
13759,Sakura-sou no Pet na Kanojo,1,0,0,0,0,0,0,0,1,...,0,1,0,0,0.817656,0.311414,0.111745,0.4,0.0,0.012658
14227,Tonari no Kaibutsu-kun,1,0,0,0,0,0,0,1,1,...,0,0,1,0,0.732272,0.272720,0.027641,0.4,0.0,0.006604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58606,Girls & Panzer: Saishuushou Part 4 Specials,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0.707670,0.000709,0.000009,0.0,1.0,0.000550
58006,Science SARU x MBS Original Short Anime Daisak...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.529667,0.000122,0.000000,0.0,1.0,0.001651
58229,Harutsugeuo to Fuuraibou,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0.000000,0.000048,0.000000,0.0,1.0,0.000000


Display the similarity when compared against another entry

In [4]:
df_Similarity = pd.DataFrame(cosine_similarity(USM.drop(columns=['Title'])), 
                             index=USM.index,
                             columns=USM.index)

df_Similarity

MAL_id,14719,13601,14741,13759,14227,14513,14345,13125,14467,14713,...,58592,57834,55806,58226,58015,58606,58006,58229,58605,58308
MAL_id,,,,,,,,,,,,,,,,,,,,,
14719,1.000000,0.256828,0.172602,0.158842,0.233755,0.383190,0.347854,0.221072,0.389183,0.258115,...,0.287624,0.350093,0.155982,0.097551,0.350357,0.109256,0.122223,0.143871,0.088102,0.136487
13601,0.256828,1.000000,0.176324,0.162546,0.179140,0.281754,0.415680,0.457785,0.329260,0.267947,...,0.172258,0.170049,0.167871,0.105812,0.170458,0.119362,0.132837,0.076288,0.093432,0.072372
14741,0.172602,0.176324,1.000000,0.483827,0.476353,0.251082,0.226342,0.317320,0.292777,0.420557,...,0.267584,0.188568,0.340732,0.372417,0.265943,0.382208,0.248993,0.086441,0.105867,0.082004
13759,0.158842,0.162546,0.483827,1.000000,0.372000,0.287232,0.207715,0.291309,0.336332,0.273431,...,0.175905,0.173764,0.312159,0.263181,0.174167,0.274142,0.323448,0.078704,0.096391,0.074664
14227,0.233755,0.179140,0.476353,0.372000,1.000000,0.387115,0.232721,0.259011,0.301179,0.561702,...,0.356124,0.274332,0.272370,0.385541,0.354680,0.395017,0.256761,0.089825,0.110011,0.085214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58606,0.109256,0.119362,0.382208,0.274142,0.395017,0.200427,0.112445,0.130023,0.233488,0.275146,...,0.427162,0.328859,0.616230,0.569750,0.425593,1.000000,0.436891,0.216286,0.264895,0.307781
58006,0.122223,0.132837,0.248993,0.323448,0.256761,0.328404,0.126714,0.144840,0.386611,0.230124,...,0.512164,0.510627,0.390261,0.558349,0.510942,0.436891,1.000000,0.399026,0.488705,0.252366
58229,0.143871,0.076288,0.086441,0.078704,0.089825,0.159962,0.077078,0.083534,0.095415,0.078836,...,0.394995,0.395911,0.198383,0.329311,0.395732,0.216286,0.399026,1.000000,0.544331,0.421637


## Test on Jujutsu Kaisen 
Show similarity features that JJK needs

In [5]:
i = USM.query('Title == "Jujutsu Kaisen"').index[0]
i = df_Similarity.loc[i].sort_values(ascending=False)[:11]

df_Results = USM.loc[i.keys()].copy()
df_Results.insert(0, 'Similarity', i)

colMask = df_Results.iloc[0].apply(lambda x : x != 0)
df_Results.loc[:, colMask]

,Similarity,Title,Is_Finished,Genre_Action,Genre_Supernatural,Genre_Award Winning,Demo_Shounen,Theme_School,Studio_MAPPA,Producer_Shueisha,...,Licensor_VIZ Media,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Members,Favorites,Related_Entries,Year,Episodes
MAL_id,,,,,,,,,,,,,,,,,,,,,
40748,1.000000,Jujutsu Kaisen,1,1,1,1,1,1,1,1,...,1,1,1,1,0.892909,0.665196,0.396664,0.8,0.666667,0.012658
51009,0.908482,Jujutsu Kaisen 2nd Season,1,1,1,0,1,1,1,1,...,0,1,1,1,0.920405,0.278167,0.102411,0.8,0.916667,0.012108
48561,0.819761,Jujutsu Kaisen 0 Movie,1,1,1,0,1,1,1,1,...,0,0,1,1,0.869754,0.264022,0.045475,0.8,0.750000,0.000000
56243,0.708690,Jujutsu Kaisen 2nd Season Recaps,1,1,1,0,1,1,1,0,...,0,0,1,1,0.761216,0.006859,0.000831,0.0,0.916667,0.000550
38000,0.657504,Kimetsu no Yaiba,1,1,1,1,1,0,0,1,...,0,1,1,1,0.874096,0.779528,0.394367,0.8,0.583333,0.013759
44511,0.650291,Chainsaw Man,1,1,0,0,1,0,1,0,...,0,1,1,1,0.876990,0.411451,0.212899,0.4,0.833333,0.006054
49926,0.648977,Kimetsu no Yaiba: Mugen Ressha-hen,1,1,1,0,1,0,0,1,...,0,1,1,1,0.861071,0.207605,0.019010,0.8,0.750000,0.003302
41467,0.644868,Bleach: Sennen Kessen-hen,1,1,1,0,1,0,0,1,...,1,1,1,1,0.955137,0.149575,0.093686,0.8,0.833333,0.006604
51019,0.626949,Kimetsu no Yaiba: Katanakaji no Sato-hen,1,1,1,0,1,0,0,1,...,0,1,1,1,0.837916,0.232567,0.040722,0.8,0.916667,0.005504


Most similarities occur on Supernatural and Action genres.

Shounen is the shared demographic. 

Shared rating is R17+

## Compare Similarity to Watched List
Test on watching the Entire JJK Series

In [6]:
df_History = USM[USM['Title'].str.contains(r'Jujutsu Kaisen', case=False, na=False, regex=True)]
df_History.Title

MAL_id
40748                      Jujutsu Kaisen
48561              Jujutsu Kaisen 0 Movie
51009           Jujutsu Kaisen 2nd Season
56243    Jujutsu Kaisen 2nd Season Recaps
Name: Title, dtype: object

#### Similar Features Shared between All Watched Entries

In [7]:
importance = df_History.drop(columns=['Title']).mean(axis=0).values
pd.Series(importance, index=USM.drop(columns=['Title']).columns).sort_values(ascending=False).head(10)

Is_Finished                              1.000000
Genre_Action                             1.000000
Producer_TOHO animation                  1.000000
Genre_Supernatural                       1.000000
Theme_School                             1.000000
Studio_MAPPA                             1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Demo_Shounen                             1.000000
Score                                    0.861071
dtype: float64

In [8]:
importance = importance.reshape([1,-1])
colMask = np.concatenate([np.array([[True, True]]), importance > 0.25], axis=1).flatten()

In [9]:
ids = pd.Series(cosine_similarity(importance, USM.drop(columns='Title')).flatten(), index=USM.index)
ids = ids.sort_values(ascending=False)
ids = ids.drop(df_History.index)    # Remove already watched

df_Results = USM.loc[ids.keys()].copy()
df_Results.insert(1, 'Similarity', ids.values)
df_Results.loc[:, colMask].head()

,Title,Similarity,Is_Finished,Genre_Action,Genre_Supernatural,Demo_Shounen,Theme_School,Studio_MAPPA,Producer_Shueisha,Producer_TOHO animation,Producer_Mainichi Broadcasting System,Producer_dugout,Producer_Sumzap,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Members,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,,
49926,Kimetsu no Yaiba: Mugen Ressha-hen,0.692428,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.861071,0.207605,0.8,0.750000
44511,Chainsaw Man,0.691091,1,1,0,1,0,1,0,0,0,1,0,1,1,1,0.876990,0.411451,0.4,0.833333
46569,Jigokuraku,0.671486,1,1,1,1,0,1,0,0,0,0,0,1,1,1,0.823444,0.193034,0.4,0.916667
51019,Kimetsu no Yaiba: Katanakaji no Sato-hen,0.670428,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.837916,0.232567,0.8,0.916667
55701,Kimetsu no Yaiba: Hashira Geiko-hen,0.669966,1,1,1,1,0,0,1,0,0,0,0,1,1,1,0.826339,0.144093,0.8,1.000000


Once again, Demon Slayer and Chainsaw Man are recommended.

# More Tests

In [10]:
class Recommender:
    def __init__(self, isPrintWatched = True, isPrintSimilarity = True):
        self.isPrintWatched = isPrintWatched
        self.isPrintSimilarity = isPrintSimilarity
        
        self.df_History = None
        self.importance = None
        self.df_Results = None
    
    def QueryWatchedDF(self, watchedRegex : str, USM : pd.DataFrame):
        self.df_History = USM[USM['Title'].str.contains(watchedRegex, case=False, na=False, regex=True)]
        
        if self.isPrintWatched:
            print('Watched:')
            print(self.df_History.Title)

    def CalcWatchedSimilarity(self, USM):
        self.importance = self.df_History.drop(columns=['Title']).mean(axis=0).values
        
        if self.isPrintSimilarity:
            print('\nKey Similarities:')
            print(pd.Series(self.importance, index=self.df_History.drop(columns=['Title']).columns).sort_values(ascending=False).head(10))
        
        self.importance = self.importance.reshape([1,-1])
    
    def Recommend(self, watchedRegex : str, USM : pd.DataFrame):
        self.QueryWatchedDF(watchedRegex, USM)
        self.CalcWatchedSimilarity(USM)
    
    
        ids = pd.Series(cosine_similarity(self.importance, USM.drop(columns='Title')).flatten(), index=USM.index)
        ids = ids.sort_values(ascending=False)
        ids = ids.drop(self.df_History.index)    # Remove already watched
        
        self.df_Results = USM.loc[ids.keys()].copy()
        self.df_Results.insert(1, 'Similarity', ids.values)
    
    def GenerateUSMTopFeaturesWeight(self, USM : pd.DataFrame, weightMultiplier = 1):
        if self.importance is None:
            raise ValueError('Recommend first to get important feature values')
        
        top_10_features_indices = self.importance.flatten().argsort()[-10:][::-1] + 1 # Offset by 1 bcuz of 'Title'
        size = top_10_features_indices.size

        USM_weighted = USM.copy()


        prevM = size + 1
        for i in np.arange(0, size):
            curr = self.importance.flatten()[top_10_features_indices[i]]
            prev = self.importance.flatten()[top_10_features_indices[i-1]]
            
            if i > 0 and curr == prev:
                m = prevM
            else:
                m = (prevM - 1) 
                prevM = m

            USM_weighted.iloc[:,top_10_features_indices[i]] = USM.iloc[:,top_10_features_indices[i]] * m * weightMultiplier

            
        
        return USM_weighted
    
    def Show(self, cutoff = 5):
        print(f'\nTop {cutoff} Mean Similarity: ', self.df_Results['Similarity'].head(cutoff).mean())
        
        colMask = np.concatenate([np.array([[True, True]]), self.importance > 0.25], axis=1).flatten()
        return self.df_Results.loc[:, colMask].head(cutoff)

## Modifiying USM
Before proceeding, we will change the USM features. This modification is done because we will be scaling the most important weights, hence having redundant info taking up the top features will be a waste.

1. Remove Is_Finished. It's the most recurring value since most anime are already completed.
2. Remove Producer & Licensor, since most viewers often prioritize only Studio. The only exception, is when the Licensor is Netflix.

In [11]:
def FindColsWithPrefix(arr, prefix):
    indices = set()
    for i, string in enumerate(arr):
        if string.startswith(prefix):
            indices.add(string)
    return indices

cols = {'Is_Finished'} | FindColsWithPrefix(USM.columns, 'Producer') | FindColsWithPrefix(USM.columns, 'Licensor')

USM = USM.drop(columns=list(cols))

## Testing Importance Weights

### Popular Fantasy: Re:Zero

### Ver 1. Original

In [12]:
r = Recommender()

In [13]:
r.Recommend(r'Re:Zero', USM)
r.Show()

Watched:
MAL_id
31240                Re:Zero kara Hajimeru Isekai Seikatsu
33142                     Re:Zero kara Hajimeru Break Time
36286    Re:Zero kara Hajimeru Isekai Seikatsu - Memory...
39921    Re:Zero kara Hajimeru Isekai Seikatsu - Memory...
38414    Re:Zero kara Hajimeru Isekai Seikatsu - Hyouke...
41590    Re:Zero kara Hajimeru Isekai Seikatsu - Hyouke...
42364          Re:Zero kara Hajimeru Break Time 2nd Season
39587     Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season
42203    Re:Zero kara Hajimeru Isekai Seikatsu 2nd Seas...
54857     Re:Zero kara Hajimeru Isekai Seikatsu 3rd Season
60012          Re:Zero kara Hajimeru Break Time 3rd Season
Name: Title, dtype: object

Key Similarities:
Source_Light novel     1.000000
Score                  0.772925
Genre_Fantasy          0.727273
Year                   0.628788
Theme_Isekai           0.545455
Studio_White Fox       0.545455
Related_Entries        0.509091
Genre_Drama            0.454545
Genre_Suspense         0.454545
T

,Title,Similarity,Genre_Suspense,Genre_Comedy,Genre_Drama,Genre_Fantasy,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Light novel,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,
48897,Overlord: Ple Ple Pleiades 4,0.718989,0,1,0,1,0,1,0,0,1,0,0,1,0,1,0.620839,0.4,0.833333
52461,Rougo ni Sonaete Isekai de 8-manmai no Kinka w...,0.711479,0,0,0,1,0,1,0,0,0,0,1,1,0,1,0.657019,0.4,0.916667
33674,No Game No Life: Zero,0.699872,0,0,1,1,0,1,0,0,0,1,0,1,0,1,0.835022,0.8,0.416667
42429,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.687483,0,0,0,1,0,1,0,0,0,0,1,1,0,1,0.820550,0.8,0.833333
45486,Kuma Kuma Kuma Bear Punch!,0.686018,0,1,0,1,0,1,0,0,0,0,1,1,0,1,0.706223,0.8,0.916667


### Ver 2. Scaling the Previously Discovered Most Important Features

In [14]:
r.isPrintWatched = False
r.Recommend(r'Re:Zero', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)


Key Similarities:
Source_Light novel                  10.000000
Score                                6.956322
Genre_Fantasy                        5.818182
Year                                 4.401515
Theme_Isekai                         3.272727
Studio_White Fox                     3.272727
Related_Entries                      2.545455
Rating_PG-13 - Teens 13 or older     1.818182
Theme_Psychological                  1.363636
Studio_Studio PuYUKAI                1.363636
dtype: float64

Top 10 Mean Similarity:  0.9415380672663911


,Title,Similarity,Genre_Suspense,Genre_Comedy,Genre_Drama,Genre_Fantasy,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Light novel,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,,,,,,
38659,Shinchou Yuusha: Kono Yuusha ga Ore Tueee Kuse...,0.948245,0,1,0,8,0,6,0,6,0,0,1,0,1,10,6.603473,2.0,4.083333
46971,Arifureta Shokugyou de Sekai Saikyou: Prologue,0.945881,0,0,0,8,0,6,0,6,0,0,0,4,0,10,5.574530,2.0,4.666667
36882,Arifureta Shokugyou de Sekai Saikyou,0.943617,0,0,0,8,0,6,0,6,0,0,1,4,0,10,5.626628,4.0,4.083333
40083,Arifureta Shokugyou de Sekai Saikyou Specials,0.942381,0,0,0,8,0,6,0,6,0,0,0,4,0,10,5.639653,2.0,4.083333
42429,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.941410,0,0,0,8,0,6,0,0,0,0,1,4,0,10,7.384949,4.0,5.833333
40815,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.940921,0,0,0,8,0,6,0,0,0,0,1,4,0,10,7.397974,4.0,4.666667
39468,Honzuki no Gekokujou: Shisho ni Naru Tame ni w...,0.938791,0,0,0,8,0,6,0,0,0,0,1,4,0,10,7.267728,4.0,4.083333
38040,Kono Subarashii Sekai ni Shukufuku wo! Movie: ...,0.938348,0,1,0,8,0,6,0,0,0,1,0,4,0,10,7.840810,4.0,4.083333
49458,Kono Subarashii Sekai ni Shukufuku wo! 3,0.938067,0,1,0,8,0,6,0,0,0,0,1,4,0,10,7.762663,4.0,7.000000


### Ver 3. Try removing the Top 5 Key Features

In [15]:
r = Recommender(False, True)
r.Recommend(r'Re:Zero', USM.drop(columns=['Source_Light novel', 'Related_Entries', 'Score', 'Related_Entries', 'Year']))
r.Show()


Key Similarities:
Genre_Fantasy                       0.727273
Studio_White Fox                    0.545455
Theme_Isekai                        0.545455
Theme_Psychological                 0.454545
Studio_Studio PuYUKAI               0.454545
Genre_Suspense                      0.454545
Genre_Drama                         0.454545
Rating_PG-13 - Teens 13 or older    0.454545
Theme_Time Travel                   0.363636
Type_Movie                          0.363636
dtype: float64

Top 5 Mean Similarity:  0.6257514950994215


,Title,Similarity,Genre_Suspense,Genre_Comedy,Genre_Drama,Genre_Fantasy,Theme_Psychological,Theme_Isekai,Theme_Time Travel,Studio_White Fox,Studio_Studio PuYUKAI,Type_Movie,Type_TV,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity)
MAL_id,,,,,,,,,,,,,,,
38472,Isekai Quartet,0.633069,0,1,0,1,0,1,0,0,1,0,1,1,0
39988,Isekai Quartet 2,0.632260,0,1,0,1,0,1,0,0,1,0,1,1,0
41567,Isekai Quartet Movie: Another World,0.631330,0,1,0,1,0,1,0,0,1,1,0,1,0
30901,Utawarerumono: Itsuwari no Kamen,0.616194,0,0,1,1,0,0,0,1,0,0,1,1,0
41379,Kimi wa Kanata,0.615904,0,0,1,1,0,1,0,0,0,1,0,1,0


### Profiling Studio Trigger

In [16]:
r = Recommender()
r.Recommend(r'Kill la Kill|Darling in the FranXX|Cyberpunk|Witch Academia', USM)
r.Show()

Watched:
MAL_id
18679                                    Kill la Kill
14349                           Little Witch Academia
21659                           Kill la Kill Specials
19489    Little Witch Academia: Mahoujikake no Parade
33489                      Little Witch Academia (TV)
35849                           Darling in the FranXX
42310                          Cyberpunk: Edgerunners
Name: Title, dtype: object

Key Similarities:
Studio_Trigger                      1.000000
Score                               0.786645
Theme_School                        0.714286
Genre_Fantasy                       0.714286
Genre_Comedy                        0.714286
Genre_Action                        0.571429
Genre_Adventure                     0.428571
Type_TV                             0.428571
Rating_PG-13 - Teens 13 or older    0.428571
Year                                0.333333
dtype: float64

Top 5 Mean Similarity:  0.6818083360006365


,Title,Similarity,Genre_Action,Genre_Adventure,Genre_Sci-Fi,Genre_Comedy,Genre_Fantasy,Genre_Ecchi,Theme_School,Theme_Urban Fantasy,Studio_Trigger,Type_Movie,Type_TV,Rating_G - All Ages,Rating_PG-13 - Teens 13 or older,Score,Year
MAL_id,,,,,,,,,,,,,,,,,
34834,Hina Logi: From Luck & Logic,0.699643,1,0,0,1,1,0,1,0,0,0,1,0,1,0.613603,0.416667
40060,BNA,0.687611,1,0,0,0,1,0,0,1,1,0,1,0,1,0.714906,0.666667
31442,Musaigen no Phantom World,0.676036,1,0,0,1,1,1,1,1,0,0,1,0,1,0.643994,0.333333
37657,Gakuen Basara,0.673144,1,0,0,1,0,0,1,0,0,0,1,0,1,0.551375,0.500000
32681,Uchuu Patrol Luluco,0.672608,1,1,1,1,0,0,0,0,1,0,1,0,1,0.740955,0.333333


### Mixed Bag: Fate Franchise
* Main - Fantasy, action
* Ilya - Magical girls, ecchi
* Grand Carnival - Comedy

In [17]:
r = Recommender()
r.Recommend(r'Fate/', USM)
r.Show()

Watched:
MAL_id
11741                                 Fate/Zero 2nd Season
13263             Fate/Zero: Onegai! Einzbern Soudanshitsu
13183                                      Fate/Zero Remix
19109              Fate/kaleid liner Prisma☆Illya Specials
14829                       Fate/kaleid liner Prisma☆Illya
19165                                       Fate/Zero Cafe
22297               Fate/stay night: Unlimited Blade Works
27821      Fate/stay night: Unlimited Blade Works Prologue
25011        Fate/kaleid liner Prisma☆Illya 2wei! Specials
20509                 Fate/kaleid liner Prisma☆Illya 2wei!
18851    Fate/kaleid liner Prisma☆Illya: Undoukai de Da...
31389    Fate/stay night: Unlimited Blade Works 2nd Sea...
31056    Fate/kaleid liner Prisma☆Illya 2wei Herz! Spec...
28701    Fate/stay night: Unlimited Blade Works 2nd Season
27525            Fate/kaleid liner Prisma☆Illya 2wei Herz!
26057    Fate/kaleid liner Prisma☆Illya 2wei!: Mahou Sh...
34321                        Fate/Grand 

,Title,Similarity,Genre_Action,Genre_Comedy,Genre_Fantasy,Studio_SILVER LINK.,Studio_ufotable,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
48585,Black Clover: Mahou Tei no Ken,0.760365,1,1,1,0,0,1,0,1,0.808973,0.4,0.916667
53865,Yozakura-san Chi no Daisakusen,0.755345,1,1,0,1,0,1,0,1,0.745297,0.4,1.000000
38268,Hangyakusei Million Arthur 2nd Season,0.752363,1,1,1,0,0,1,0,0,0.562952,0.4,0.583333
45654,Xiyou Ji: Zai Shi Yao Wang,0.751725,1,0,1,0,0,1,0,0,0.630970,0.0,0.750000
37555,Hangyakusei Million Arthur,0.745717,1,1,1,0,0,1,0,0,0.522431,0.4,0.500000


In [18]:
r.isPrintWatched = False
r.Recommend(r'Fate/', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)


Key Similarities:
Rating_PG-13 - Teens 13 or older    7.111111
Score                               6.226628
Genre_Action                        5.155556
Genre_Fantasy                       4.622222
Genre_Comedy                        3.733333
Year                                3.020370
Source_Manga                        2.400000
Related_Entries                     1.688889
Studio_SILVER LINK.                 1.333333
Studio_ufotable                     1.066667
dtype: float64

Top 10 Mean Similarity:  0.9587718536916101


,Title,Similarity,Genre_Action,Genre_Comedy,Genre_Fantasy,Studio_SILVER LINK.,Studio_ufotable,Rating_PG-13 - Teens 13 or older,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
39551,Tensei shitara Slime Datta Ken 2nd Season,0.965286,8,8,8,0,0,10,0,6,7.762663,4.0,5.250000
41487,Tensei shitara Slime Datta Ken 2nd Season Part 2,0.964937,8,8,8,0,0,10,0,6,7.697540,4.0,5.250000
48585,Black Clover: Mahou Tei no Ken,0.963132,8,8,8,0,0,10,0,6,7.280753,2.0,6.416667
37430,Tensei shitara Slime Datta Ken,0.962682,8,8,8,0,0,10,0,6,7.476122,2.0,3.500000
52211,Mashle,0.956971,8,8,8,0,0,10,0,6,6.798842,2.0,6.416667
48643,Koi wa Sekai Seifuku no Ato de,0.956674,8,8,8,0,0,10,0,6,6.499276,2.0,5.833333
55813,Mashle: Shinkakusha Kouho Senbatsu Shiken-hen,0.955878,8,8,8,0,0,10,0,6,7.046310,4.0,7.000000
53580,Tensei shitara Slime Datta Ken 3rd Season,0.955268,8,8,8,0,0,10,0,6,6.876990,4.0,7.000000
37986,Trinity Seven Movie 2: Heavens Library to Crim...,0.954622,8,8,8,0,0,10,0,6,6.421129,2.0,4.083333


### Fighting + Gore: Baki

In [19]:
r = Recommender()
r.Recommend(r'Baki:|^Baki', USM)
r.Show()

Watched:
MAL_id
33566    Baki: Most Evil Death Row Convicts Special Anime
34443                                                Baki
39555                             Baki: Dai Raitaisai-hen
42940                             Hanma Baki: Son of Ogre
51318                  Hanma Baki: Son of Ogre 2nd Season
Name: Title, dtype: object

Key Similarities:
Theme_Gore                               1.000000
Demo_Shounen                             1.000000
Theme_Combat Sports                      1.000000
Genre_Sports                             1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Studio_TMS Entertainment                 0.800000
Type_ONA                                 0.800000
Score                                    0.725904
Related_Entries                          0.720000
dtype: float64

Top 5 Mean Similarity:  0.6183861697739805


,Title,Similarity,Genre_Sports,Demo_Shounen,Theme_Gore,Theme_Combat Sports,Studio_TMS Entertainment,Type_ONA,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
59497,Rising Impact Season 2,0.639765,1,1,0,0,0,1,0,1,0.804631,0.4,1.000000
57487,Rising Impact,0.638312,1,1,0,0,0,1,0,1,0.743849,0.4,1.000000
35789,Yowamushi Pedal: Glory Line,0.607003,1,1,0,0,1,0,0,1,0.749638,0.8,0.500000
31783,Yowamushi Pedal: New Generation,0.603861,1,1,0,0,1,0,0,1,0.756874,0.8,0.416667
54730,Kinnikuman: Kanpeki Chоujin Shiso-hen,0.602990,1,1,0,1,0,0,0,1,0.638205,0.8,1.000000


In [20]:
r.isPrintWatched = False
r.Recommend(r'Baki:|^Baki', r.GenerateUSMTopFeaturesWeight(USM))
r.Show()


Key Similarities:
Theme_Gore                               10.000000
Demo_Shounen                             10.000000
Theme_Combat Sports                      10.000000
Genre_Sports                             10.000000
Rating_R - 17+ (violence & profanity)    10.000000
Source_Manga                             10.000000
Type_ONA                                  7.200000
Studio_TMS Entertainment                  6.400000
Score                                     5.081331
Related_Entries                           4.320000
dtype: float64

Top 5 Mean Similarity:  0.7704267034876453


,Title,Similarity,Genre_Sports,Demo_Shounen,Theme_Gore,Theme_Combat Sports,Studio_TMS Entertainment,Type_ONA,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
54730,Kinnikuman: Kanpeki Chоujin Shiso-hen,0.772421,10,10,0,10,0,0,0,10,4.467438,4.8,1.000000
51009,Jujutsu Kaisen 2nd Season,0.770712,0,10,10,0,0,0,10,10,6.442836,4.8,0.916667
37007,Hinomaruzumou,0.769777,10,10,0,10,0,0,0,10,5.196816,2.4,0.500000
49376,Mou Ippon!,0.769644,10,10,0,10,0,0,0,10,4.751085,2.4,0.916667
52505,Dark Gathering,0.769580,0,10,10,0,0,0,10,10,5.510854,2.4,0.916667


In [21]:
r = Recommender()
r.Recommend(r'Jojo', USM)

r.isPrintWatched = False
r.Recommend(r'Jojo', r.GenerateUSMTopFeaturesWeight(USM))
r.Show(10)

Watched:
MAL_id
14719                        JoJo no Kimyou na Bouken (TV)
20899    JoJo no Kimyou na Bouken Part 3: Stardust Crus...
26055    JoJo no Kimyou na Bouken Part 3: Stardust Crus...
31933    JoJo no Kimyou na Bouken Part 4: Diamond wa Ku...
37991       JoJo no Kimyou na Bouken Part 5: Ougon no Kaze
38972    JoJo no Kimyou na Bouken Part 5: Ougon no Kaze...
48661         JoJo no Kimyou na Bouken Part 6: Stone Ocean
53273    JoJo no Kimyou na Bouken Part 6: Stone Ocean P...
51367    JoJo no Kimyou na Bouken Part 6: Stone Ocean P...
Name: Title, dtype: object

Key Similarities:
Studio_David Production                  1.000000
Genre_Action                             1.000000
Genre_Adventure                          1.000000
Demo_Shounen                             1.000000
Rating_R - 17+ (violence & profanity)    1.000000
Source_Manga                             1.000000
Theme_Super Power                        0.888889
Score                                    0.834539
Related

,Title,Similarity,Genre_Action,Genre_Adventure,Demo_Shounen,Theme_Super Power,Studio_David Production,Type_ONA,Type_TV,Rating_R - 17+ (violence & profanity),Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,,
39489,Spriggan (ONA),0.961687,9,7,8,0,8,1,0,10,8,3.219971,1.6,0.833333
52741,Undead Unluck,0.942047,9,0,8,6,8,0,3,10,8,3.878437,1.6,1.000000
33191,Kishibe Rohan wa Ugokanai,0.907343,9,0,8,0,8,0,0,10,8,3.769899,1.6,0.416667
53998,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,0.893319,9,7,8,0,0,0,3,10,8,4.558611,3.2,0.916667
41467,Bleach: Sennen Kessen-hen,0.893224,9,7,8,0,0,0,3,10,8,4.775687,3.2,0.833333
56784,Bleach: Sennen Kessen-hen - Soukoku-tan,0.893210,9,7,8,0,0,0,3,10,8,4.573082,3.2,1.000000
32370,D.Gray-man Hallow,0.890878,9,7,8,0,0,0,3,10,8,3.820550,1.6,0.333333
46569,Jigokuraku,0.889128,9,7,8,0,0,0,3,10,8,4.117221,1.6,0.916667
42627,Peach Boy Riverside,0.888634,9,7,8,0,0,0,3,10,8,2.778582,1.6,0.750000


### Niche Cute Girls Doing Cute Things: Yuru Camp

In [22]:
r = Recommender()
r.Recommend(r'Yuru Camp|Heya Camp', USM)
r.Show()

Watched:
MAL_id
37341                          Yuru Camp△ Specials
34798                                   Yuru Camp△
41061    Heya Camp△: Sauna to Gohan to Sanrin Bike
38476                                   Heya Camp△
49026                 Yuru Camp△ Season 2 Specials
38474                          Yuru Camp△ Season 2
38475                             Yuru Camp△ Movie
53410                          Yuru Camp△ Season 3
58855                 Yuru Camp△ Season 3 Specials
Name: Title, dtype: object

Key Similarities:
Genre_Slice of Life                 1.000000
Theme_CGDCT                         1.000000
Rating_PG-13 - Teens 13 or older    1.000000
Source_Manga                        1.000000
Theme_Iyashikei                     0.888889
Score                               0.788873
Studio_C-Station                    0.777778
Year                                0.740741
Type_Special                        0.444444
Type_TV                             0.444444
dtype: float64

Top 5 Mean Si

,Title,Similarity,Genre_Slice of Life,Theme_CGDCT,Theme_Iyashikei,Studio_C-Station,Type_Special,Type_TV,Rating_PG-13 - Teens 13 or older,Source_Manga,Score,Related_Entries,Year
MAL_id,,,,,,,,,,,,,
27887,Yama no Susume Second Season Specials,0.850799,1,1,1,0,1,0,1,1,0.646889,0.0,0.166667
48491,Yama no Susume: Next Summit,0.840154,1,1,1,0,0,1,1,1,0.758321,0.4,0.833333
35672,Yama no Susume Third Season,0.821299,1,1,1,0,0,1,1,1,0.752533,0.8,0.500000
45425,Slow Loop,0.811151,1,1,1,0,0,1,1,1,0.701881,0.4,0.833333
39808,Non Non Biyori Nonstop,0.806778,1,1,1,0,0,1,1,1,0.862518,0.8,0.750000


# Summary

The original content-based recommender is able to provide similar anime based on their features. However if we want a more specific target, we could scale the key features to retrieve only the highest matching features.

We also removed noisy and redundant features such as 'Is_Finished', 'Producer_', 'Licensor_'.

# Bag of Words
Another approach is creating features from the synopsis. We extract important words and measure its frequency.

In [23]:
df_BOW = pd.read_csv('Cleaned.csv', index_col=0)[['Title', 'Synopsis']]
df_BOW = df_BOW.loc[~df_BOW.index.duplicated(keep='first')] # Duplicates present from shows continued airing between multiple seasons but still classified as one.
df_BOW = df_BOW.dropna()

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
pd.set_option('display.max_colwidth', None)

matrix_BOW = TfidfVectorizer(stop_words='english').fit_transform(df_BOW['Synopsis'])
cosine_sim = cosine_similarity(matrix_BOW)

def BOW_Recommend(item_index, cosine_sim=cosine_sim):
    similarity_scores = list(enumerate(cosine_sim[item_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similar_items = [x[0] for x in similarity_scores]
    
    return similar_items


### Test Dandandan
Not very good.

It focuses on the characters names: Momo and Ken.

In [25]:
i = df_BOW.reset_index().query('Title == "Dandadan"').index[0]
recommended_items = BOW_Recommend(i)
df_BOW.iloc[recommended_items[:10]]

,Title,Synopsis
MAL_id,,
57334,Dandadan,"Reeling from her recent breakup, Momo Ayase, a popular high schooler, shows kindness to her socially awkward schoolmate, Ken Takakura, by standing up to his bullies. Ken misunderstands her intentions, believing he has made a new friend who shares his obsession with aliens and UFOs. However, Momo's own eccentric occult beliefs lie in the supernatural realm; she thinks aliens do not exist. A rivalry quickly brews as each becomes determined to prove the other wrong. Despite their initial clash over their opposing beliefs, Momo and Ken form an unexpected but intimate friendship, a bond forged in a series of supernatural battles and bizarre encounters with urban legends and paranormal entities. As both develop unique superhuman abilities, they learn to supplement each other's weaknesses, leading them to wonder if their newfound partnership may be about more than just survival."
30014,Momokuri,"After taking one hundred secret photos and observing him from afar for months, second-year high schooler Yuki Kurihara has finally mustered up the courage to ask out her first-year crush Shinya ""Momo"" Momotsuki. Although taken by surprise, the bashful Momo accepts; however, he does not know the profoundly abnormal truth. As her strait-laced friend, Norika Mizuyama, has observed, Yuki has developed some unnerving—but nonetheless sincere—habits: taking pictures of Momo in secret, doing extensive research into his personal life, collecting his used straws, and even going ""Momo watching."" Though Momo remains blissfully unaware of his new girlfriend's peculiar habits, he does notice some oddities in their daily conversations. Still unsure and nervous about his first relationship, Momo finds himself regularly getting into awkward interactions due to his inexperience, but nevertheless resolves to make his new girlfriend happy. Momokuri follows Yuki and Momo as they shyly explore their newfound love, and also deal with the problems that arise from it."
10389,Momo e no Tegami,"After the unexpected death of her father, 11-year-old Momo Miyaura leaves Tokyo with her mother and moves to an old remote island in Seto Inland Sea. The only memento she has from her father is an unfinished letter with only two words inside: ""Dear Momo""—along with her heart's unrest from it. In the new and unfamiliar small town, Momo reluctantly tries to adjust to the outmoded wooden buildings, silent crop fields, and mysterious isolated shrines. One day, while exploring the attic of her new home, she finds a worn out picture book about youkai. Following this discovery, strange things begin to happen around town, and Momo is greeted by the arrival of three troublesome youkai. Momo e no Tegami tells the story of a young girl as she struggles to adapt to her bizarre new life and ultimately come to terms with her father's mysterious letter."
23171,Mahou Shoujo wa Kiss Shite Kawaru,After Iori fell in love with Ken at a young age she wanted to be his wife more than anything. Ken feels the same way about Iori. When Ken is nearly killed by a monster she is told the only way to save him is to have sex with other men to make her more powerful as a magical girl.
38561,Pan to Boku no Momo-chan,"The story follows a boy named Shiro and his aunt, Momo, who live together after the death of Shiro's mother, Momo's older sister. Shiro makes bread for Momo in the mornings."
51635,Kenda Master Ken,"Kenda Master Ken chronicles the exploits of Ken Tamaki as he battles rivals using kendama, a traditional Japanese skill toy."
51636,Kenda Master Ken (TV),"Ken Tamaki defeats corrupt organizations and even finds love through his favorite pastime, the Japanese skill toy ""kendama."" But kendama is no mere toy. The stakes are high and Ken will have to fight for his right to play in KENDA MASTER KEN!"
39518,Vampire in the Garden,"During a winter long ago, near-immortal vampires began plaguing the world. As their population grew at an astounding rate, they promp

### Test Frieren
It's quite good. It's fixating on `long time ago` followed by `but now settled peaceful`.

In [26]:
i = df_BOW.reset_index().query('Title == "Sousou no Frieren"').index[0]
recommended_items = BOW_Recommend(i)
df_BOW.iloc[recommended_items[:10]]

,Title,Synopsis
MAL_id,,
52991,Sousou no Frieren,"During their decade-long quest to defeat the Demon King, the members of the hero's party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them. However, the time that Frieren spends with her comrades is equivalent to merely a fraction of her life, which has lasted over a thousand years. When the party disbands after their victory, Frieren casually returns to her ""usual"" routine of collecting spells across the continent. Due to her different sense of time, she seemingly holds no strong feelings toward the experiences she went through. As the years pass, Frieren gradually realizes how her days in the hero's party truly impacted her. Witnessing the deaths of two of her former companions, Frieren begins to regret having taken their presence for granted; she vows to better understand humans and create real personal connections. Although the story of that once memorable journey has long ended, a new tale is about to begin."
56885,Sousou no Frieren: ●● no Mahou,"Frieren loves collecting peculiar spells, and her ever-growing arsenal of magic contains the perfect spells for many occasions. Whether it be by stealthily removing alcohol from Heiter's drinks or by remedying her own sleep issues, Frieren is sure to brighten her companions' days with her magic."
38062,Endro~!,"In a world of adventurers and magic lies Naral Island. Every generation, a Demon Lord rises to plague the land, and every generation, a Hero is born to subdue him. For countless centuries, the cycle has repeated with no end in sight. The latest Hero, Juulia ""Yusha"" Charldetto, has almost completed her valiant campaign alongside her party members: responsible priest Seiran ""Seira"" Élénoir, enigmatic mage Meiza ""Mei"" Endust, and hyper-energetic warrior Fai Fai. In the final battle against the Demon Lord, Yusha's party attempt a risky spell to cast their enemy into the drifts of time. But the incantation goes awry, sending Yusha and her friends back to a time before the Demon Lord, before Yusha becomes the Hero, and before the party had even graduated as adventurers. With their memories of the future erased, the four girls restart their ambitions to become the Hero's Party, aspiring to defeat the Demon Lord. However, in a sudden twist of fate, the Demon Lord was also sent back in time with her memories intact. Reduced to the form of a little girl, the Demon Lord takes the name Mao and infiltrates the adventurers' school as a teacher, planning to stop Yusha before she becomes a hero. Thus begins the story of Yusha and her friends, in their quest to defeat the Demon Lord, not knowing that the one they seek is right by their side."
34028,Idol Jihen,"Increasing income divide, creeping environmental pollution, unsolvable waste issues, childcare waiting lists being discussed without those concerned, repeated corruption… The government, smeared by vested interests, can't do a thing against the many problems and sources of discontent. It's in this situation, with Japan cornered with no way out, that idols rise up to save the day! The Heroine Party, Sunlight Party, Starlight Party, Bishoujo Party, Wakaba Party, Subculture New Party, and SOS Party. From these seven idol political parties, the idols who have become National Diet members and representatives for each prefecture will smash through the sense of stagnation covering Japan using the power of song and dance! They'll bring back the smiling faces of the people, and wrap Japan in a glittering aura!!"
18677,Yuusha ni Narenakatta Ore wa Shibushibu Shuushoku wo Ketsui Shimashita.,"Dreaming of becoming a hero and vanquishing the Demon King, Raul Chaser enters the Hero Training Program in pursuit of his ambition. However, when the Demon King is defeated and peace returns to the world, the Hero Training Program is suspended indefinitely, makin